# Lesson 05 — 用固定资源处理连续数据

这是一段真实的 C++ 实验：你修改代码格，再运行，就会看到下方深色虚拟设备的输入、输出和检查结果变化。Python 只负责让 VS Code Notebook 启动 C++ 编译和实验台。

## 先运行

按顺序运行初始化格和后面的代码格。绿色 ✓ 表示本格 C++ 符合当前契约；红色 ✗ 会给出预期与实际输出。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


嵌入式设备不能假设内存无限。一个不断接收采样数据的工具需要明确容量、错误和数据所有权。

## 最小知识

`std::array` 给出编译期固定容量；`enum`/状态字段让调用者知道满了还是可用。这里输入协议是 `push:数字`、`average`、`clear`，缓存最多四个样本。


## 1. 第一个实验（Guided）

先运行。不要急着读每一行；先操作实验台，观察输入与输出之间的关系。


In [ ]:
%%academy_lab 05-embedded-cpp guided guided pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text.rfind("push:", 0) == 0) {
                ++b_.output.queue_depth;
            }
        }

    private:
        Device& b_;
    };
}


## 2. 修改一次

把容量从 4 改为 2，再推入三个样本，观察为什么调用方必须看到 error。

重新运行这一个格，再在实验台确认你的预测。


In [ ]:
%%academy_lab 05-embedded-cpp modify
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text.rfind("push:", 0) == 0) {
                ++b_.output.queue_depth;
            }
        }

    private:
        Device& b_;
    };
}


## 3. 故障实验（Debug）

下面实现不断追加而不检查容量；在嵌入式系统里，这种“能一直放”的假设通常不成立。运行后失败是预期现象。用面板的时间线、波形、事务或终端找出失败发生在何时。


In [ ]:
%%academy_lab 05-embedded-cpp broken exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() {
            if (b_.input.text.rfind("push:", 0) == 0) {
                ++b_.output.queue_depth;
            }
        }

    private:
        Device& b_;
    };
}


## 4. 你来完成（Build）

用固定数组保存不超过四个整数。满时拒绝输入并设置 `output.error`；`average` 计算已有数据平均值。

默认这格会失败，失败信息是需求的一部分。先自己尝试；需要时再看 [分层 Hint](../lessons/05-embedded-cpp/hints.md)。


In [ ]:
%%academy_lab 05-embedded-cpp exercise exercise fail
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& b) : b_(b) {
        }
        void tick() { // TODO: 固定容量 4，解析 push / average / clear
            b_.output.state = "ready";
        }

    private:
        Device& b_;
    };
}


## 5. 开放挑战

把命令解析与缓存类拆分为头文件/源文件。为无效文本、负数和清空空缓存决定错误策略。

## 你刚刚真正学到了什么？

资源限制需要进入接口契约。容量、错误状态和数据格式都比“看起来能跑”更早被明确。

## 硬件迁移

STEP 2 可将 `push` 换成真实传感器采样。缓存仍可固定；是否需要动态内存取决于上限能否证明。

完成自己的尝试后才查看 [独立 Solution Notebook](solutions/05-embedded-cpp-solution.ipynb)。
